# 01 · Exploração dos dados de F1

**Objetivo:** carregar os resultados de uma corrida e de uma classificação (GP de São Paulo 2025) para entender a estrutura dos dados do FastF1 antes de construir o modelo.

**Perguntas que quero responder:**
- Quais colunas existem e quais são úteis para prever a posição final?
- Como aparecem os abandonos (DNF)?
- O que significa `GridPosition = 0`?
- Como estão guardados os tempos de Q1, Q2 e Q3?
- Qual coluna usar como chave para juntar corrida e classificação?

## 1. Setup
Importa o FastF1 e ativa o cache local, para não baixar os mesmos dados a cada execução.

In [48]:
import fastf1
import pandas as pd

In [49]:
fastf1.Cache.enable_cache('../cache')

## 2. Corrida
Resultados da corrida: posição final, grid de largada, status e pontos.

In [50]:

corrida = fastf1.get_session(2025, "São Paulo Grand Prix", "R")

# Não vamos trazer os dados de volta a volta, nem velocidade, nem clima e sem mensagens de direção da prova
corrida.load(laps=False, telemetry=False, weather=False, messages=False)

core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core           INFO 	Finished loading data for 20 drivers: ['4', '12', '1', '63', '81', '87', '30', '6', '27', '10', '23', '31', '55', '14', '43', '18', '22', '44', '16', '5']


In [51]:
resultado_corrida = corrida.results

In [61]:
# vamos verificar o tamanho do dataframe
resultado_corrida.shape
resultado_corrida.columns

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps'],
      dtype='object')

In [54]:
# Verificando colunas que só existem depois da corrida (Vazamento de dados)
resultado_corrida[["Abbreviation", "Position", "Status", "Laps", "Time"]]

,Abbreviation,Position,Status,Laps,Time
DriverNumber,,,,,
4,NOR,1.0,Finished,71.0,0 days 01:32:01.596000
12,ANT,2.0,Finished,71.0,0 days 00:00:10.388000
1,VER,3.0,Finished,71.0,0 days 00:00:10.750000
63,RUS,4.0,Finished,71.0,0 days 00:00:15.267000
81,PIA,5.0,Finished,71.0,0 days 00:00:15.749000
87,BEA,6.0,Finished,71.0,0 days 00:00:29.630000
30,LAW,7.0,Finished,71.0,0 days 00:00:52.642000
6,HAD,8.0,Finished,71.0,0 days 00:00:52.873000
27,HUL,9.0,Finished,71.0,0 days 00:00:53.324000


**O que observei:**
- `Position` → posição final de cada piloto na corrida. É a variável que queremos prever (**alvo**).
- `Status` → indica se o piloto completou a prova (`Finished`) ou abandonou (`Retired`).
- `Laps` → número de voltas completadas. Quem abandonou tem menos que as 71 voltas da prova (Hamilton saiu na volta 37, Bortoleto na volta 0).
- `Time` → para o vencedor, é o tempo total de prova; para os demais, é a diferença para o vencedor. Quem abandonou fica com `NaT` (tempo nulo).
- `Abbreviation` → sigla do piloto (NOR, VER...). Serve apenas para identificação.

**Conclusão:** `Status`, `Laps` e `Time` só são conhecidos **depois** da bandeirada. Se forem usados como variáveis, o modelo "cola" a resposta (**vazamento de dados**): acerta no treino, mas não funciona na prática, porque essas informações não existem antes da largada. Por isso, essas colunas **ficam de fora** do modelo.

In [55]:
resultado_corrida[["Abbreviation", "GridPosition", "Position", "ClassifiedPosition", "Points", "Q1"]]

,Abbreviation,GridPosition,Position,ClassifiedPosition,Points,Q1
DriverNumber,,,,,,
4,NOR,1.0,1.0,1,25.0,NaT
12,ANT,2.0,2.0,2,18.0,NaT
1,VER,19.0,3.0,3,15.0,NaT
63,RUS,6.0,4.0,4,12.0,NaT
81,PIA,4.0,5.0,5,10.0,NaT
87,BEA,8.0,6.0,6,8.0,NaT
30,LAW,7.0,7.0,7,6.0,NaT
6,HAD,5.0,8.0,8,4.0,NaT
27,HUL,10.0,9.0,9,2.0,NaT


**O que observei:**
- `GridPosition` → posição de largada, definida na classificação de sábado
- `ClassifiedPosition` → classificação oficial da prova. Igual a `Position` para quem terminou, mas quem abandonou recebe `R` (*Retired*) em vez de um número.
- `Points` → pontos ganhos na corrida. Só do 1º ao 10º lugar (25, 18, 15, 12, 10, 8, 6, 4, 2, 1).
- `Q1` → vazia (`NaT`) em todas as linhas, porque os tempos de classificação só vêm preenchidos na sessão de classificação (`"Q"`), não na de corrida.

**Conclusão:**
- `GridPosition` é conhecida **antes** da largada → ✅ **pode ser usada** e deve ser uma das variáveis mais importantes.
- `ClassifiedPosition` e `Points` só existem **depois** da corrida → ❌ **vazamento de dados**, ficam de fora. (Os pontos acumulados em corridas **anteriores**, porém, poderão virar feature.)
- Os tempos de Q1, Q2 e Q3 seriam úteis, mas precisam ser carregados da sessão de classificação, que é o próximo passo.

## 3. Classificação
Resultados da quali: posição e tempos de Q1, Q2 e Q3.

In [56]:
# Carregando a classificação (sábado), só com os resultados
quali = fastf1.get_session(2025, "São Paulo Grand Prix", "Q")
quali.load(laps=False, telemetry=False, weather=False, messages=False)
resultado_quali = quali.results

core           INFO 	Loading data for São Paulo Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core           INFO 	Finished loading data for 19 drivers: ['4', '12', '16', '81', '6', '63', '30', '87', '10', '27', '14', '23', '44', '18', '55', '1', '31', '43', '22']


In [57]:
resultado_quali[["Abbreviation", "Position", "Q1", "Q2", "Q3"]]

,Abbreviation,Position,Q1,Q2,Q3
DriverNumber,,,,,
4,NOR,1.0,0 days 00:01:09.656000,0 days 00:01:09.616000,0 days 00:01:09.511000
12,ANT,2.0,0 days 00:01:10.192000,0 days 00:01:09.774000,0 days 00:01:09.685000
16,LEC,3.0,0 days 00:01:09.934000,0 days 00:01:09.801000,0 days 00:01:09.805000
81,PIA,4.0,0 days 00:01:09.928000,0 days 00:01:09.835000,0 days 00:01:09.886000
6,HAD,5.0,0 days 00:01:10.083000,0 days 00:01:09.970000,0 days 00:01:09.931000
63,RUS,6.0,0 days 00:01:09.935000,0 days 00:01:09.880000,0 days 00:01:09.942000
30,LAW,7.0,0 days 00:01:10.108000,0 days 00:01:09.950000,0 days 00:01:09.962000
87,BEA,8.0,0 days 00:01:09.891000,0 days 00:01:09.755000,0 days 00:01:09.977000
10,GAS,9.0,0 days 00:01:09.885000,0 days 00:01:09.857000,0 days 00:01:10.002000


**O que observei:**
- A classificação funciona como um mata-mata em três partes: os 10 primeiros têm tempo em Q1, Q2 e Q3; do 11º ao 15º, o Q3 fica vazio (eliminados no Q2); do 16º em diante, só existe o Q1 (eliminados no Q1). Os `NaT` **não são erro**, são consequência do formato.
- A posição na classificação **nem sempre é a posição de largada**: Verstappen foi 16º na quali e largou em 19º; Ocon foi 17º e largou em 20º (punições ou largada do pit lane).
- A quali tem **19 pilotos**, enquanto a corrida tem **20**.

**Conclusão:**
- Os tempos de Q1, Q2 e Q3 são conhecidos **antes** da largada → ✅ podem virar features (por exemplo, a diferença para a pole).
- `Position` da quali e `GridPosition` da corrida são informações diferentes, e as duas podem ser úteis.
- Antes de juntar as tabelas, preciso descobrir qual piloto está faltando na quali.

### Verificação: piloto ausente na classificação
A corrida tem 20 pilotos e a quali tem 19. Antes de juntar as duas tabelas, vou descobrir quem está faltando.

In [58]:
faltando = ~resultado_corrida["Abbreviation"].isin(resultado_quali["Abbreviation"])
resultado_corrida[faltando]

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
DriverNumber,,,,,,,,,,,,,,,,,,,,,
5,5,,BOR,bortoleto,Sauber,,sauber,Gabriel,Bortoleto,Gabriel Bortoleto,...,20.0,R,18.0,NaT,NaT,NaT,NaT,Retired,0.0,0.0


**Resultado:** o piloto ausente na classificação é **Gabriel Bortoleto** (BOR, Sauber). Ele não tem registro na quali, mas largou a corrida (grid 18º) e abandonou sem completar nenhuma volta.


In [59]:
# Selecionando só as colunas novas e renomeia position para não confundir com a posição da corrida
quali_enxuta = resultado_quali[["DriverId", "Position", "Q1", "Q2", "Q3"]].rename(
    columns={"Position": "QualiPosition"}
)

quali_enxuta.head()

,DriverId,QualiPosition,Q1,Q2,Q3
DriverNumber,,,,,
4,norris,1.0,0 days 00:01:09.656000,0 days 00:01:09.616000,0 days 00:01:09.511000
12,antonelli,2.0,0 days 00:01:10.192000,0 days 00:01:09.774000,0 days 00:01:09.685000
16,leclerc,3.0,0 days 00:01:09.934000,0 days 00:01:09.801000,0 days 00:01:09.805000
81,piastri,4.0,0 days 00:01:09.928000,0 days 00:01:09.835000,0 days 00:01:09.886000
6,hadjar,5.0,0 days 00:01:10.083000,0 days 00:01:09.970000,0 days 00:01:09.931000


In [ ]:
# Remove Q1/Q2/Q3 da corrida (sempre vazias) e faz o LEFT JOIN com a quali pelo DriverId
corrida_sem_quali = resultado_corrida.drop(columns=["Q1", "Q2", "Q3"])
base = pd.merge(corrida_sem_quali, quali_enxuta, on="DriverId", how="left")
base.shape

(20, 23)

## 4. Conclusões

**1. Colunas úteis para o modelo**
- 🎯 Alvo: `Position` (posição final).
- ✅ Podem ser usadas (existem antes da largada): `GridPosition`, `QualiPosition`, `Q1`, `Q2`, `Q3`.
- ❌ Não podem (só existem depois da corrida): `Status`, `Laps`, `Time`, `ClassifiedPosition`, `Points`.

**2. Abandonos (DNF)**
Aparecem como `Retired` em `Status` e `R` em `ClassifiedPosition`.

**3. `GridPosition = 0`**
Não aconteceu nesta corrida: quem largou do pit lane aparece no fim do grid.

**4. Tempos de Q1, Q2 e Q3**
Guardados como tempo (`timedelta`). Ficam vazios para quem foi eliminado antes daquela parte da classificação.

**5. Chave para juntar as tabelas**
`DriverId`, com LEFT JOIN, para não perder pilotos sem quali (caso do Bortoleto).

**Resultado:** tabela `base` com 20 pilotos e 23 colunas, juntando corrida e classificação.